# LeetCode 2022 — Convert 1D Array Into 2D Array bằng Python

**Link đề:** https://leetcode.com/problems/convert-1d-array-into-2d-array/
**Độ khó:** Easy · **Chủ đề:** Array, Matrix, Simulation

```python
class Solution:
    def construct2DArray(self, original: List[int], m: int, n: int) -> List[List[int]]:

```

## Mục tiêu học
1. Hiểu đề: xếp `original` thành ma trận `m` hàng × `n` cột theo thứ tự **từng hàng** (row-major); không đủ/thừa phần tử thì trả `[]`.
2. Nắm điều kiện khả thi **`m * n == len(original)`** và lý do.
3. Hiểu **list lồng nhau** (list of lists), chỉ số `grid[r][c]`, slicing, `range(0, L, n)`, `divmod`.
4. Viết 4 cách: con trỏ `idx`, cắt lát theo hàng, công thức `i → (i // n, i % n)`, list comprehension — trace từng bước.
5. Tránh bẫy kinh điển `[[0] * n] * m` (các hàng **dùng chung** một list), test và làm bài tập biến thể (reshape, transpose).

## 1. Đề bài yêu cầu gì?

Cho mảng 1 chiều `original` và hai số nguyên `m`, `n`. Tạo mảng 2 chiều `m` hàng, `n` cột dùng **tất cả** phần tử của `original`:
- Phần tử chỉ số `0 .. n−1` tạo **hàng đầu**,
- chỉ số `n .. 2n−1` tạo **hàng thứ hai**, v.v.

Trả về mảng `m × n`, hoặc **mảng 2D rỗng `[]`** nếu không thể.

| `original` | `m` | `n` | Kết quả | Lý do |
|---|:-:|:-:|---|---|
| `[1,2,3,4]` | 2 | 2 | `[[1,2],[3,4]]` | 4 = 2 × 2 |
| `[1,2,3]` | 1 | 3 | `[[1,2,3]]` | 3 = 1 × 3 |
| `[1,2]` | 1 | 1 | `[]` | 2 ≠ 1 × 1 |

**Constraints:** `1 <= original.length <= 5·10^4`, `1 <= original[i] <= 10^5`, `1 <= m, n <= 4·10^4`.

Chú ý: `m * n` có thể lên tới `1.6·10^9` — **lớn hơn nhiều** độ dài mảng; phải kiểm tra khả thi trước khi tạo ma trận.

In [1]:
original, m, n = [1, 2, 3, 4, 5, 6], 2, 3
print("len =", len(original), "| m * n =", m * n)
print("hàng 0:", original[0:3])
print("hàng 1:", original[3:6])

len = 6 | m * n = 6
hàng 0: [1, 2, 3]
hàng 1: [4, 5, 6]


## 2. Nền tảng Python cần biết

### 2.1. List lồng nhau — ma trận trong Python

Ma trận là **list chứa các list** (mỗi list con là một hàng). `grid[r]` là hàng `r`, `grid[r][c]` là phần tử hàng `r`, cột `c`.

In [2]:
grid = [[1, 2, 3],
        [4, 5, 6]]
print("số hàng:", len(grid), "| số cột:", len(grid[0]))
print("grid[1]    =", grid[1])
print("grid[1][2] =", grid[1][2])
for row in grid:
    print(row)

số hàng: 2 | số cột: 3
grid[1]    = [4, 5, 6]
grid[1][2] = 6
[1, 2, 3]
[4, 5, 6]


### 2.2. Thứ tự **hàng trước** (row-major)

Đọc ma trận theo hàng từ trái sang phải, hết hàng xuống hàng tiếp — giống đọc sách. Phần tử thứ `i` (chỉ số trong mảng 1D) nằm ở:

$$r = i \,//\, n, \qquad c = i \,\%\, n$$

Ví dụ `n = 3`: `i = 4` → `r = 1`, `c = 1`.

In [3]:
n = 3
for i in range(6):
    print(f"i = {i} -> (hàng {i // n}, cột {i % n})")

i = 0 -> (hàng 0, cột 0)
i = 1 -> (hàng 0, cột 1)
i = 2 -> (hàng 0, cột 2)
i = 3 -> (hàng 1, cột 0)
i = 4 -> (hàng 1, cột 1)
i = 5 -> (hàng 1, cột 2)


### 2.3. `divmod(a, b)` trả `(a // b, a % b)`

In [4]:
print(divmod(4, 3))
r, c = divmod(5, 3)
print("r =", r, "c =", c)

(1, 1)
r = 1 c = 2


### 2.4. `range(0, L, n)` — bước nhảy `n`

`range(0, 6, 3)` → `0, 3`: chỉ số **bắt đầu** của từng hàng. Kết hợp slicing `original[s:s+n]` lấy cả hàng.

In [5]:
original = [1, 2, 3, 4, 5, 6]
for s in range(0, len(original), 3):
    print("bắt đầu", s, "->", original[s:s + 3])

bắt đầu 0 -> [1, 2, 3]
bắt đầu 3 -> [4, 5, 6]


### 2.5. Bẫy `[[0] * n] * m` — các hàng dùng chung một list!

`[0] * n` tạo một list mới. Nhưng `[row] * m` tạo list chứa **m tham chiếu tới cùng một `row`** — sửa một hàng thì mọi hàng đổi theo. Cách đúng: list comprehension `[[0] * n for _ in range(m)]` tạo `m` list **riêng**.

In [6]:
bad = [[0] * 3] * 2
bad[0][0] = 99
print("sai :", bad, "| cùng object?", bad[0] is bad[1])

good = [[0] * 3 for _ in range(2)]
good[0][0] = 99
print("đúng:", good, "| cùng object?", good[0] is good[1])

sai : [[99, 0, 0], [99, 0, 0]] | cùng object? True
đúng: [[99, 0, 0], [0, 0, 0]] | cùng object? False


Toán tử `is` kiểm tra hai biến có trỏ tới **cùng một object** trong bộ nhớ không (khác `==` chỉ so giá trị). Biến `_` là quy ước cho "biến không dùng tới".

## 3. Điều kiện khả thi — bước bắt buộc đầu tiên

Ma trận `m × n` có đúng `m · n` ô. Đề yêu cầu dùng **tất cả** phần tử và lấp **đầy** ma trận:
- `len(original) < m·n` → thiếu phần tử, không lấp đầy được.
- `len(original) > m·n` → thừa phần tử, không dùng hết được.

→ Khả thi **khi và chỉ khi** `m * n == len(original)` (đúng như hint 2).

In [7]:
for L, m, n in [(4, 2, 2), (3, 1, 3), (2, 1, 1), (6, 4, 2), (8, 2, 3)]:
    print(f"len = {L}, m = {m}, n = {n}: m*n = {m*n} ->", "khả thi" if m * n == L else "trả []")

len = 4, m = 2, n = 2: m*n = 4 -> khả thi
len = 3, m = 1, n = 3: m*n = 3 -> khả thi
len = 2, m = 1, n = 1: m*n = 1 -> trả []
len = 6, m = 4, n = 2: m*n = 8 -> trả []
len = 8, m = 2, n = 3: m*n = 6 -> trả []


## 4. Cách 1 — Trực giác: con trỏ `idx` chạy qua `original`

Đúng như hint 3: giữ chỉ số `idx` của **phần tử tiếp theo** cần đặt vào ma trận. Tạo từng hàng bằng cách lấy `n` phần tử liên tiếp.

```python
if m * n != len(original):
    return []
result = []
idx = 0
for r in range(m):
    row = []
    for c in range(n):
        row.append(original[idx])
        idx += 1
    result.append(row)
return result
```

In [8]:
def construct_pointer(original, m, n):
    if m * n != len(original):
        return []
    result = []
    idx = 0
    for r in range(m):
        row = []
        for c in range(n):
            row.append(original[idx])
            idx += 1
        result.append(row)
    return result

print(construct_pointer([1, 2, 3, 4], 2, 2))
print(construct_pointer([1, 2, 3], 1, 3))
print(construct_pointer([1, 2], 1, 1))

[[1, 2], [3, 4]]
[[1, 2, 3]]
[]


## 5. TRACE từng bước (Cách 1)

Ví dụ `original = [1, 2, 3, 4, 5, 6]`, `m = 2`, `n = 3`.

| r | c | idx | original[idx] | row sau append | result sau hàng |
|:-:|:-:|:-:|:-:|---|---|
| 0 | 0 | 0 | 1 | [1] | |
| 0 | 1 | 1 | 2 | [1, 2] | |
| 0 | 2 | 2 | 3 | [1, 2, 3] | [[1,2,3]] |
| 1 | 0 | 3 | 4 | [4] | |
| 1 | 1 | 4 | 5 | [4, 5] | |
| 1 | 2 | 5 | 6 | [4, 5, 6] | [[1,2,3],[4,5,6]] |

Invariant dễ thấy: `idx == r * n + c` ở mọi bước.

In [9]:
def trace_construct(original, m, n):
    print(f"original = {original}, m = {m}, n = {n}")
    if m * n != len(original):
        print(f"  m*n = {m*n} != len = {len(original)} -> trả []")
        return []
    result, idx = [], 0
    for r in range(m):
        row = []
        for c in range(n):
            row.append(original[idx])
            print(f"  r={r}, c={c}, idx={idx} (= r*n+c = {r*n+c}) -> lấy {original[idx]}, row = {row}")
            idx += 1
        result.append(row)
        print(f"  => xong hàng {r}: result = {result}")
    return result

trace_construct([1, 2, 3, 4, 5, 6], 2, 3)
print()
trace_construct([1, 2], 1, 1)

original = [1, 2, 3, 4, 5, 6], m = 2, n = 3
  r=0, c=0, idx=0 (= r*n+c = 0) -> lấy 1, row = [1]
  r=0, c=1, idx=1 (= r*n+c = 1) -> lấy 2, row = [1, 2]
  r=0, c=2, idx=2 (= r*n+c = 2) -> lấy 3, row = [1, 2, 3]
  => xong hàng 0: result = [[1, 2, 3]]
  r=1, c=0, idx=3 (= r*n+c = 3) -> lấy 4, row = [4]
  r=1, c=1, idx=4 (= r*n+c = 4) -> lấy 5, row = [4, 5]
  r=1, c=2, idx=5 (= r*n+c = 5) -> lấy 6, row = [4, 5, 6]
  => xong hàng 1: result = [[1, 2, 3], [4, 5, 6]]

original = [1, 2], m = 1, n = 1
  m*n = 1 != len = 2 -> trả []


[]

## 6. Nút thắt / cải tiến: lấy cả hàng một lần bằng slicing

Cách 1 đúng và `O(m·n)`, nhưng vòng trong `append` từng phần tử khá dài dòng. Quan sát: hàng `r` chính là **đoạn liên tiếp** `original[r*n : (r+1)*n]`. Slicing lấy cả đoạn trong một lệnh (bên trong vẫn là `O(n)`, nhưng chạy bằng C nên nhanh hơn vòng Python).

## 7. Cách 2 — Cắt lát theo hàng (lời giải chính)

In [10]:
def construct_slice(original, m, n):
    if m * n != len(original):
        return []
    return [original[r * n:(r + 1) * n] for r in range(m)]

print(construct_slice([1, 2, 3, 4], 2, 2))
print(construct_slice([1, 2, 3], 1, 3))
print(construct_slice([1, 2], 1, 1))
print(construct_slice([1, 2, 3, 4, 5, 6], 3, 2))

[[1, 2], [3, 4]]
[[1, 2, 3]]
[]
[[1, 2], [3, 4], [5, 6]]


Giảng từng đoạn:
- `if m * n != len(original): return []` — kiểm tra khả thi **trước**.
- `for r in range(m)` — mỗi hàng.
- `original[r * n:(r + 1) * n]` — `n` phần tử bắt đầu từ `r * n` (chỉ số `(r+1)*n` **không** lấy).
- List comprehension gom các hàng thành ma trận. Mỗi slice là **list mới** → không gặp bẫy dùng chung.

Viết tương đương bằng `range(0, len(original), n)`:

In [11]:
def construct_slice_step(original, m, n):
    if m * n != len(original):
        return []
    return [original[s:s + n] for s in range(0, len(original), n)]

print(construct_slice_step([1, 2, 3, 4, 5, 6], 2, 3))

[[1, 2, 3], [4, 5, 6]]


## 8. Code nộp LeetCode

In [12]:
from typing import List

class Solution:
    def construct2DArray(self, original: List[int], m: int, n: int) -> List[List[int]]:
        if m * n != len(original):
            return []
        return [original[r * n:(r + 1) * n] for r in range(m)]

for original, m, n, expected in [([1, 2, 3, 4], 2, 2, [[1, 2], [3, 4]]),
                                 ([1, 2, 3], 1, 3, [[1, 2, 3]]),
                                 ([1, 2], 1, 1, [])]:
    got = Solution().construct2DArray(original, m, n)
    print(original, m, n, "->", got)
    assert got == expected

[1, 2, 3, 4] 2 2 -> [[1, 2], [3, 4]]
[1, 2, 3] 1 3 -> [[1, 2, 3]]
[1, 2] 1 1 -> []


## 9. Cách 3 — Tạo ma trận rỗng rồi đặt từng phần tử bằng `divmod`

Ngược với Cách 1 (đi theo ô ma trận), cách này đi theo **phần tử của `original`**: phần tử `i` thuộc ô `(i // n, i % n)`. Rất hữu ích khi cần chuyển đổi qua lại giữa chỉ số 1D và 2D (ví dụ trong bài BFS trên lưới, "flatten" ma trận).

In [13]:
def construct_divmod(original, m, n):
    if m * n != len(original):
        return []
    grid = [[0] * n for _ in range(m)]       # m hàng RIÊNG BIỆT
    for i, x in enumerate(original):
        r, c = divmod(i, n)
        grid[r][c] = x
    return grid

print(construct_divmod([1, 2, 3, 4, 5, 6], 2, 3))
print(construct_divmod([1, 2, 3, 4, 5, 6], 3, 2))

[[1, 2, 3], [4, 5, 6]]
[[1, 2], [3, 4], [5, 6]]


## 10. Cách 4 — Dùng iterator: `zip(*[iter(original)] * n)` (mẹo Python nâng cao)

`iter(original)` tạo **một** iterator. `[it] * n` là list chứa `n` tham chiếu tới **cùng** iterator đó (ở đây việc dùng chung lại là **cố ý**!). `zip(*...)` mỗi lần lấy một phần tử từ mỗi tham chiếu — nhưng vì là cùng iterator, nó lấy `n` phần tử **liên tiếp** → đúng một hàng.

Mẹo này để mở rộng hiểu biết; khi đi phỏng vấn, Cách 2 rõ ràng hơn.

In [14]:
it = iter([1, 2, 3, 4, 5, 6])
print(next(it), next(it))      # next() lấy phần tử tiếp theo của iterator
print(list(it))                # phần còn lại

def construct_iter(original, m, n):
    if m * n != len(original):
        return []
    it = iter(original)
    return [list(row) for row in zip(*[it] * n)]

print(construct_iter([1, 2, 3, 4, 5, 6], 2, 3))
print(construct_iter([1, 2, 3, 4], 4, 1))

1 2
[3, 4, 5, 6]
[[1, 2, 3], [4, 5, 6]]
[[1], [2], [3], [4]]


## 11. Vì sao đúng?

**Điều kiện:** đã chứng minh ở mục 3 — khả thi ⟺ `m·n = L` (L = độ dài). Khi khả thi, cách xếp là **duy nhất** (đề quy định rõ đoạn nào vào hàng nào), nên chỉ cần dựng đúng quy tắc.

**Cách 2:** hàng `r` theo đề gồm chỉ số `r·n .. r·n + n − 1` = `original[r*n:(r+1)*n]`. Với `r` từ `0` tới `m − 1`, các đoạn này **rời nhau** và **phủ kín** `0 .. m·n − 1 = 0 .. L − 1` → dùng mỗi phần tử đúng một lần.

**Cách 1/3:** song ánh `i ↔ (i // n, i % n)` giữa `{0..L−1}` và các ô `{0..m−1} × {0..n−1}` (vì `i = r·n + c` với `0 ≤ c < n` là phép chia có dư duy nhất). Invariant Cách 1: `idx = r·n + c`.

In [15]:
m, n = 3, 4
cells = {divmod(i, n) for i in range(m * n)}
print("số ô khác nhau:", len(cells), "= m*n =", m * n)
print("ngược lại r*n + c:", [r * n + c for r, c in sorted(cells)][:6], "...")

số ô khác nhau: 12 = m*n = 12
ngược lại r*n + c: [0, 1, 2, 3, 4, 5] ...


## 12. Time / Space và bảng so sánh

Gọi `L = len(original)`. Khi khả thi `L = m·n`.

| Cách | Ý tưởng | Time | Space (ngoài output) | Ghi chú |
|---|---|---|---|---|
| 1. Con trỏ `idx` | `append` từng phần tử | `O(L)` | `O(1)` | Theo hint, dễ hiểu |
| 2. Slicing theo hàng | `original[r*n:(r+1)*n]` | `O(L)` | `O(1)` | **Khuyến nghị** |
| 3. `divmod` | `grid[i//n][i%n] = x` | `O(L)` | `O(1)` | Học ánh xạ 1D ↔ 2D |
| 4. `zip(*[iter]*n)` | iterator dùng chung | `O(L)` | `O(1)` | Mẹo nâng cao |

Output tự nó tốn `O(L)`. Khi **không** khả thi, mọi cách trả `[]` ngay trong `O(1)` — quan trọng vì `m·n` có thể tới `1.6·10^9`: nếu tạo ma trận trước rồi mới kiểm tra sẽ **tràn bộ nhớ**.

## 13. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

1. Ba ví dụ đề.
2. Một hàng (`m = 1`), một cột (`n = 1`).
3. Thiếu phần tử (`m·n > L`) và thừa phần tử (`m·n < L`).
4. `m`, `n` rất lớn (`4·10^4 × 4·10^4`) với mảng nhỏ → phải trả `[]` ngay.
5. Kiểm tra các hàng **độc lập** (sửa một hàng không ảnh hưởng hàng khác).
6. Đối chiếu ngẫu nhiên giữa các cách.

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS {actual if len(str(actual)) < 40 else '(...)'}")

f = construct_slice
check("đề 1", f([1, 2, 3, 4], 2, 2), [[1, 2], [3, 4]])
check("đề 2", f([1, 2, 3], 1, 3), [[1, 2, 3]])
check("đề 3", f([1, 2], 1, 1), [])
check("một cột", f([1, 2, 3], 3, 1), [[1], [2], [3]])
check("thiếu phần tử", f([1, 2, 3], 2, 2), [])
check("thừa phần tử", f([1, 2, 3, 4, 5], 2, 2), [])
check("m, n cực lớn", f([1, 2, 3], 40000, 40000), [])
g = f([1, 2, 3, 4], 2, 2)
g[0][0] = 99
check("các hàng độc lập", g[1], [3, 4])

đề 1                       PASS [[1, 2], [3, 4]]
đề 2                       PASS [[1, 2, 3]]
đề 3                       PASS []
một cột                    PASS [[1], [2], [3]]
thiếu phần tử              PASS []
thừa phần tử               PASS []
m, n cực lớn               PASS []
các hàng độc lập           PASS [3, 4]


In [17]:
import random

rng = random.Random(2022)
for _ in range(2000):
    L = rng.randint(1, 24)
    original = [rng.randint(1, 100) for _ in range(L)]
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    exp = construct_pointer(original, m, n)
    for fn in (construct_slice, construct_slice_step, construct_divmod, construct_iter):
        assert fn(original, m, n) == exp, (fn.__name__, original, m, n)
    assert Solution().construct2DArray(original, m, n) == exp
print("2000 test ngẫu nhiên: các cách khớp nhau")

2000 test ngẫu nhiên: các cách khớp nhau


## 14. Lỗi người mới hay gặp

### Lỗi 1: Quên kiểm tra khả thi

In [18]:
def wrong_no_check(original, m, n):
    return [original[r * n:(r + 1) * n] for r in range(m)]

print("thiếu:", wrong_no_check([1, 2, 3], 2, 2))      # [[1, 2], [3]] — hàng 2 thiếu
print("thừa :", wrong_no_check([1, 2, 3, 4, 5], 2, 2)) # bỏ sót 5
print("đúng :", construct_slice([1, 2, 3], 2, 2), construct_slice([1, 2, 3, 4, 5], 2, 2))

thiếu: [[1, 2], [3]]
thừa : [[1, 2], [3, 4]]
đúng : [] []


**Vì sao sai:** slicing **không báo lỗi** khi vượt biên — nó lặng lẽ trả đoạn ngắn hơn hoặc rỗng. **Sửa:** `if m * n != len(original): return []`.

### Lỗi 2: Kiểm tra bằng `<` thay vì `!=`

In [19]:
def wrong_less_only(original, m, n):
    if len(original) < m * n:          # quên trường hợp thừa
        return []
    return [original[r * n:(r + 1) * n] for r in range(m)]

print("sai :", wrong_less_only([1, 2, 3, 4, 5], 2, 2))
print("đúng:", construct_slice([1, 2, 3, 4, 5], 2, 2))

sai : [[1, 2], [3, 4]]
đúng: []


**Sửa:** đề yêu cầu dùng **tất cả** phần tử → phải bằng đúng.

### Lỗi 3: Bẫy `[[0] * n] * m`

In [20]:
def wrong_shared_rows(original, m, n):
    if m * n != len(original):
        return []
    grid = [[0] * n] * m               # m tham chiếu tới CÙNG một hàng
    for i, x in enumerate(original):
        grid[i // n][i % n] = x
    return grid

print("sai :", wrong_shared_rows([1, 2, 3, 4], 2, 2))   # [[3, 4], [3, 4]]
print("đúng:", construct_divmod([1, 2, 3, 4], 2, 2))

sai : [[3, 4], [3, 4]]
đúng: [[1, 2], [3, 4]]


**Vì sao sai:** mọi hàng là một object → lần ghi cuối đè lên tất cả. **Sửa:** `[[0] * n for _ in range(m)]`.

### Lỗi 4: Nhầm `i // m` / `i % m` (chia cho số hàng thay vì số cột)

In [21]:
def wrong_div_by_m(original, m, n):
    if m * n != len(original):
        return []
    grid = [[0] * n for _ in range(m)]
    for i, x in enumerate(original):
        grid[i % m][i // m] = x        # xếp theo CỘT, không phải theo hàng
    return grid

print("sai :", wrong_div_by_m([1, 2, 3, 4, 5, 6], 2, 3))   # [[1, 3, 5], [2, 4, 6]]
print("đúng:", construct_divmod([1, 2, 3, 4, 5, 6], 2, 3))

sai : [[1, 3, 5], [2, 4, 6]]
đúng: [[1, 2, 3], [4, 5, 6]]


**Vì sao sai:** `(i % m, i // m)` là thứ tự **cột trước** (column-major). Đề yêu cầu **hàng trước** → chia cho **số cột `n`**. Mẹo nhớ: *"mỗi hàng chứa n phần tử, nên cứ n phần tử thì xuống hàng"*.

## 15. Mẹo nhận dạng pattern

- "Reshape", "chuyển 1D thành 2D", "xếp vào lưới" → kiểm tra **tổng số ô** trước, rồi ánh xạ `i ↔ (i // n, i % n)`.
- "Duyệt ma trận như một mảng phẳng" (ví dụ binary search trên ma trận — LeetCode 74) → dùng cùng công thức để không cần làm phẳng thật.
- Tạo ma trận rỗng trong Python → **luôn** `[[v] * n for _ in range(m)]`.

## 16. Biến thể

1. **LeetCode 566 — Reshape the Matrix:** ma trận `r × c` → `r' × c'`; nếu không được thì trả **ma trận gốc** (khác bài này trả `[]`).
2. **Làm phẳng (flatten)** ma trận về 1D.
3. **LeetCode 867 — Transpose Matrix.**
4. **LeetCode 74 — Search a 2D Matrix:** binary search với chỉ số ảo `mid → (mid // n, mid % n)`.

In [22]:
def flatten(grid):
    return [x for row in grid for x in row]          # comprehension 2 tầng: hàng trước, phần tử sau

def matrix_reshape(mat, r, c):                       # LC 566
    flat = flatten(mat)
    if r * c != len(flat):
        return mat
    return [flat[i * c:(i + 1) * c] for i in range(r)]

def transpose(grid):                                 # LC 867
    return [list(col) for col in zip(*grid)]

print(flatten([[1, 2], [3, 4], [5, 6]]))
print(matrix_reshape([[1, 2], [3, 4]], 1, 4), matrix_reshape([[1, 2], [3, 4]], 2, 4))
print(transpose([[1, 2, 3], [4, 5, 6]]))

[1, 2, 3, 4, 5, 6]
[[1, 2, 3, 4]] [[1, 2], [3, 4]]
[[1, 4], [2, 5], [3, 6]]


`zip(*grid)` "mở" các hàng làm đối số cho `zip`, nên `zip` ghép phần tử cùng **cột** lại với nhau — đó là chuyển vị.

## 17. Bài tập tự luyện (TODO — chưa chạy)

### Bài 1 — Tính tay
`original = [5, 6, 7, 8, 9, 10]`, `m = 3`, `n = 2`. Phần tử `9` nằm ở ô nào? Viết ma trận kết quả.

### Bài 2 — Dự đoán output
`wrong_shared_rows([1, 2, 3, 4, 5, 6], 3, 2)` trả về gì?

### Bài 3 — Tự viết lời giải slicing

In [23]:
def exercise_construct(original: list[int], m: int, n: int) -> list[list[int]]:
    # TODO 1: kiểm tra m * n == len(original)
    # TODO 2: trả list các slice original[r*n:(r+1)*n]
    pass

# assert exercise_construct([1, 2, 3, 4], 2, 2) == [[1, 2], [3, 4]]
# assert exercise_construct([1, 2], 1, 1) == []

### Bài 4 — Biến thể: LeetCode 566 — Reshape the Matrix (không được thì trả ma trận gốc)

In [24]:
def exercise_reshape(mat: list[list[int]], r: int, c: int) -> list[list[int]]:
    # TODO: flatten, kiểm tra r*c, cắt lát
    pass

# assert exercise_reshape([[1, 2], [3, 4]], 1, 4) == [[1, 2, 3, 4]]
# assert exercise_reshape([[1, 2], [3, 4]], 2, 4) == [[1, 2], [3, 4]]

### Bài 5 — Biến thể: xếp theo **cột trước** (column-major): `original[0..m-1]` là cột đầu

In [25]:
def exercise_construct_column_major(original: list[int], m: int, n: int) -> list[list[int]]:
    # TODO: phần tử i nằm ở ô (i % m, i // m)
    pass

# assert exercise_construct_column_major([1, 2, 3, 4, 5, 6], 2, 3) == [[1, 3, 5], [2, 4, 6]]

### Bài 6 — Mini challenge: LeetCode 74 — tìm `target` trong ma trận `m × n` mà khi làm phẳng là dãy tăng, **không** làm phẳng thật, `O(log(m·n))`

In [26]:
def exercise_search_matrix(matrix: list[list[int]], target: int) -> bool:
    # TODO: binary search lo = 0, hi = m*n - 1; mid -> matrix[mid // n][mid % n]
    pass

# M = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]]
# assert exercise_search_matrix(M, 3) == True
# assert exercise_search_matrix(M, 13) == False

## 18. Gợi ý, đáp án tham khảo, template và checklist

### Gợi ý 3 mức
| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Chỉ số của 9 trong `original` | `i = 4`, `n = 2` | `divmod(4, 2)` |
| 2 | Ba hàng là cùng một list | lần ghi cuối của mỗi cột thắng | cột 0 cuối cùng là 5, cột 1 là 6 |
| 3 | Khuôn mục 7 | `if m * n != len(original): return []` | list comprehension |
| 4 | `flatten` rồi như bài chính | `if r * c != len(flat): return mat` | `flat[i*c:(i+1)*c]` |
| 5 | Mỗi **cột** có `m` phần tử | `r, c = i % m, i // m` | ma trận rỗng riêng từng hàng |
| 6 | Coi ma trận như mảng phẳng | `while lo <= hi:` | `v = matrix[mid // n][mid % n]` |

### Đáp án
- Bài 1: `9` có chỉ số `4` → `divmod(4, 2) = (2, 0)` → hàng 2, cột 0. Ma trận `[[5, 6], [7, 8], [9, 10]]`.
- Bài 2: `[[5, 6], [5, 6], [5, 6]]`.

In [27]:
def answer_construct(original, m, n):
    if m * n != len(original):
        return []
    return [original[r * n:(r + 1) * n] for r in range(m)]

def answer_reshape(mat, r, c):
    flat = [x for row in mat for x in row]
    if r * c != len(flat):
        return mat
    return [flat[i * c:(i + 1) * c] for i in range(r)]

def answer_construct_column_major(original, m, n):
    if m * n != len(original):
        return []
    grid = [[0] * n for _ in range(m)]
    for i, x in enumerate(original):
        grid[i % m][i // m] = x
    return grid

def answer_search_matrix(matrix, target):
    m, n = len(matrix), len(matrix[0])
    lo, hi = 0, m * n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        v = matrix[mid // n][mid % n]
        if v == target:
            return True
        if v < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

assert divmod(4, 2) == (2, 0) and answer_construct([5, 6, 7, 8, 9, 10], 3, 2) == [[5, 6], [7, 8], [9, 10]]
assert wrong_shared_rows([1, 2, 3, 4, 5, 6], 3, 2) == [[5, 6], [5, 6], [5, 6]]
assert answer_construct([1, 2, 3, 4], 2, 2) == [[1, 2], [3, 4]] and answer_construct([1, 2], 1, 1) == []
assert answer_reshape([[1, 2], [3, 4]], 1, 4) == [[1, 2, 3, 4]]
assert answer_reshape([[1, 2], [3, 4]], 2, 4) == [[1, 2], [3, 4]]
assert answer_construct_column_major([1, 2, 3, 4, 5, 6], 2, 3) == [[1, 3, 5], [2, 4, 6]]
M = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]]
assert answer_search_matrix(M, 3) is True and answer_search_matrix(M, 13) is False
assert all(answer_search_matrix(M, v) for row in M for v in row)
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


### Template nhớ nhanh

```python
class Solution:
    def construct2DArray(self, original: List[int], m: int, n: int) -> List[List[int]]:
        if m * n != len(original):          # khả thi <=> đúng số ô
            return []
        return [original[r * n:(r + 1) * n] for r in range(m)]
```

Công thức vàng: **`i ↔ (i // n, i % n)`** và ngược lại **`r * n + c`**.

### Checklist trước khi submit
- [ ] Kiểm tra `m * n == len(original)` **trước** khi tạo ma trận (dùng `!=`, không phải `<`)?
- [ ] Mỗi hàng là list **riêng** (không dùng `[[0] * n] * m`)?
- [ ] Chia cho **số cột `n`** khi đổi chỉ số 1D → 2D?
- [ ] Không thể thì trả `[]` (không phải `[[]]` hay `None`)?
- [ ] Đã test `m = 1`, `n = 1`, thiếu và thừa phần tử?